In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import box
import sys
import numpy as np
import networkx as nx
import osmnx as ox
import matplotlib.pyplot as plt

# Import Necessary Clustering Functions

In [ ]:
sys.path.append('../src/nlr_ghe_screen/clustering/')
from seed_clustering import (
    find_seed_centroids_single
)
from construct_clusters import (
    build_clusters,
    create_cluster_subgraph,
)
from distance_matrices import (
    road_network_distance_to_centroid,
    euclidian_distance_to_centroid
)
from load_profile_processing import (
    get_building_load_profiles
)
from milp import (
    solve_cluster
)

sys.path.append('../src/nlr_ghe_screen/api/')
from building_footprint_api import (
    get_overture_footprints,
    get_ornl_footprints,
    join_ornl_overture_buildings
    
)
from osmnx_api import (
    get_road_network,
    plot_network_graph
)

# Download Necessary Building and Road Network Data

In [ ]:
TEST_BBOX = (47.58, -122.36, 47.61, -122.31) # xmin, ymin, xmax, ymax
south, west, north, east = TEST_BBOX

In [ ]:
ghe_locations = gpd.read_file("../src/exports/ghe_locations.geojson")
geom_box = box(west, south, east, north)
cropped_ghes = gpd.clip(ghe_locations, geom_box)
cropped_ghes = cropped_ghes.to_crs(cropped_ghes.estimate_utm_crs())

In [ ]:
overture_buildings = get_overture_footprints(bbox=TEST_BBOX)
ornl_buildings = get_ornl_footprints(bbox=TEST_BBOX)

In [ ]:
ornl_buildings = ornl_buildings.to_crs(cropped_ghes.crs)
overture_buildings = overture_buildings.to_crs(cropped_ghes.crs)

In [ ]:
buildings_joined = join_ornl_overture_buildings(
    ornl_data=ornl_buildings,
    overture_data=overture_buildings
)

In [ ]:
network = get_road_network(bbox=TEST_BBOX)

In [ ]:
network = network.to_undirected()
network.is_directed()

# Aggregate Cluster Seeds + Nearby Buildings

In [ ]:
cropped_ghes, seed_centroids = find_seed_centroids_single(
    ghe_locations=cropped_ghes,
)

In [ ]:
des_clusters = build_clusters(
    seed_centroids=seed_centroids,
    building_data=buildings_joined,
    G=network,
    max_distance=500,
)

In [ ]:
idx = 250
cluster = des_clusters[des_clusters['cluster'] == idx]

m = cropped_ghes[cropped_ghes['cluster'] == idx].explore(color='green')
seed_centroids[seed_centroids['cluster'] == idx].explore(m=m, color='red')
cluster.explore(m=m, color='blue')

In [ ]:
cluster_seed_centroid = seed_centroids.loc[seed_centroids['cluster'] == idx, 'geometry'].item()
cluster['centroid_distance'] = road_network_distance_to_centroid(
    buildings_df=cluster,
    centroid=cluster_seed_centroid,
    max_distance=500
)

# Download Representative Load Profile Curves

In [ ]:
building_types = cluster['comstock_type'].tolist()
building_sizes = cluster['sqfeet'].tolist()
building_distances = cluster['centroid_distance'].tolist()

In [ ]:
building_sources = [
    'restock' if building_type == "Multi-Family with 5+ Units" else 'comstock'
    for building_type in building_types
]

In [ ]:
frames = get_building_load_profiles(
    building_types=building_types,
    building_sizes=building_sizes,
    building_sources=building_sources,
    state='WA',
)

In [ ]:
for frame in frames:
    print(frame.attrs)

In [ ]:
resampled_frames = [frame.resample("D").mean() for frame in frames]
building_loads = [building['net_load'].tolist() for building in resampled_frames]

building_types.insert(0, 'Cluster Seed')
building_distances.insert(0,0)
building_loads.insert(0, [0] * 366)

fig, ax = plt.subplots(figsize=(12, 5))
colors = plt.cm.tab10.colors  
for i, (load, type) in enumerate(zip(building_loads, building_types)):
    ax.plot(
        range(len(load)),
        load,
        color=colors[i % len(colors)],
        label=type
    )

ax.set_xlabel("Time")
ax.set_ylabel("Average energy (kWh)")
ax.set_title(f"Energy Usage")
ax.legend(title=None)

plt.legend()
plt.show()

# Run MILP Optimization on the Cluster

In [ ]:
solution = solve_cluster(
    loads=building_loads,
    distances=building_distances,
    seed_idx=0,
    k_min=8,
    weight_peak=0.5,
    weight_net=0.5,
)
solution

In [ ]:
final_loads = [building_loads[i] for i in solution['selected']]
final_loads.append(solution['net_curve'])
final_types = [building_types[i] for i in solution['selected']]
final_types.append('Cluster Net')

fig, ax = plt.subplots(figsize=(12, 5))
colors = plt.cm.tab10.colors  
for i, (load, type) in enumerate(zip(final_loads, final_types)):
    ax.plot(
        range(len(load)),
        load,
        color=colors[i % len(colors)],
        label=type
    )

ax.set_xlabel("Time")
ax.set_ylabel("Average energy (kWh)")
ax.set_title(f"Energy Usage")
ax.legend(title=None)

plt.legend()
plt.show()

# Plot the Cluster

In [ ]:
from shapely.geometry import LineString, Point
import folium

def plot_network_graph(
    network: nx.MultiDiGraph, 
    extra_layers: list
):
    """
    """
    edge_lines = []
    for u, v, data in network.edges(data=True):
        if 'geometry' in data:
            geom = data['geometry']
        else:
            # Build a LineString from node coordinates
            x1, y1 = network.nodes[u]['x'], network.nodes[u]['y']
            x2, y2 = network.nodes[v]['x'], network.nodes[v]['y']
            geom = LineString([(x1, y1), (x2, y2)])
        edge_lines.append({'u': u, 'v': v, 'geometry': geom})

    gdf_edges = gpd.GeoDataFrame(edge_lines, crs=network.graph['crs'])

    # Convert NetworkX nodes to a GeoDataFrame of Points
    node_points = []
    for n, data in network.nodes(data=True):
        node_points.append(
            {'node': n, 'geometry': Point(data['x'], data['y']), **data}
        )

    gdf_nodes = gpd.GeoDataFrame(node_points, crs=network.graph['crs'])

    # 3. Create the base interactive map using GeoPandas .explore()
    # m = base_gdf.explore(color='green', name='Base Map')

    m = gdf_edges.explore(color='blue', weight=2, name='Network Edges')
    m = gdf_nodes.explore(m=m, color='red', marker_kwds={'radius': 4}, name='Network Nodes')

    for name, gdf, color in (extra_layers or []):
        m = gdf.explore(m=m, color=color, name=name)

    # Fit to everything combined, since each explore() call re-fits to its own data
    all_bounds = [gdf_edges.to_crs(4326).total_bounds]
    all_bounds += [g.to_crs(4326).total_bounds for _, g, _ in (extra_layers or [])]
    minx = min(b[0] for b in all_bounds); miny = min(b[1] for b in all_bounds)
    maxx = max(b[2] for b in all_bounds); maxy = max(b[3] for b in all_bounds)
    m.fit_bounds([[miny, minx], [maxy, maxx]])

    folium.LayerControl().add_to(m)
    return m

In [ ]:
cluster_indices = [i - 1 for i in solution['selected']]
final_cluster = cluster.reset_index(drop=True).copy()
final_cluster = final_cluster.iloc[cluster_indices[1:]]

final_seed = cropped_ghes[cropped_ghes['cluster'] == idx]

In [ ]:
mst_subgraph, distance = create_cluster_subgraph(
    cluster_buildings=final_cluster,
    cluster_ghes=final_seed,
    G=network
)

In [ ]:
m = plot_network_graph(
    mst_subgraph,
    extra_layers=[
        ('GHEs', final_seed, 'green'),
        ('Buildings', final_cluster, 'blue')
    ],
)
m